In [ ]:
import torch
import sys
import os
import xarray as xr
import numpy as np
import random
import matplotlib.pyplot as plt

In [ ]:
# Add OceanAdjoint to current path
sys.path.append("/nobackup/smousav2/adjoint_learning/Controls/OceanAdjoint/adjoint")
import post_processing
import data_loaders
import model
figpath = "/nobackup/smousav2/adjoint_learning/Controls/Plots"

In [ ]:
def load_sample_points(path_in, sample_indices, wet, device="cpu", engine="netcdf4", remove_pole=False):
    """
    Load etan_ad and forcing associated sam_i, sam_j for specific sample_point indices.

    Args:
        data_in (str): Path to NetCDF file for input data (etan_ad here)
        sample_indices (list[int]): List of sample_point indices to extract
        wet (torch.Tensor): Shape [lat, lon] wet mask
        device (str): Torch device ("cpu" or "cuda")
        engine (str): NetCDF engine (default="netcdf4")
        remove_pole (bool): Whether to remove the pole value from data_in

    Returns:
        data_in (torch.Tensor): Shape [len(sample_indices), lag_days, in_channel, lat, lon]
        sam_i (torch.Tensor): Shape [len(sample_indices)]
        sam_j (torch.Tensor): Shape [len(sample_indices)]
    """
    wet = wet.to(device)
    ds = xr.open_dataset(path_in, engine=engine)
    subset = ds.isel(sample_point=sample_indices)
    ds.close()
    data_in = subset["etan_ad"].values            # Shape: (N_targets, T, C_in, H, W)
    data_in = torch.tensor(data_in, dtype=torch.float32, device=device)
    if remove_pole:
        wet_mask = (wet > 0).to(data_in.dtype)  
        ref = data_in[:, :, :, -1, 0]  # Reference point at the pole
        data_in = data_in - ref[..., None, None] * wet_mask[None, None, None, :, :]

    # Extract sam_i and sam_j
    sam_i = torch.tensor(subset["sam_i"].values, dtype=torch.int64, device=device)
    sam_j = torch.tensor(subset["sam_j"].values, dtype=torch.int64, device=device)

    return data_in, sam_i, sam_j


In [ ]:
# === Parameters ===
C_in = 1    # only SSH
C_out = C_in
C_out_total = C_out
pred_residual = False   # Set pred_residual to True only for pred_status="state"
remove_pole = True
path_in = "/nobackupp17/ifenty/AD_ML/2025-08-30_all_training_points_10d_lag/adetan_training_points/consolidated/etan_ad_2025-08-30_3594pts_10d_consolidated.nc"
wet_mask_path = "/nobackupp17/ifenty/AD_ML/sam_grid/SAM_GRID_v01.nc"
n_unroll = 1
loss_name = "Charbonnier"  # "MSE", "Relative", "Charbonnier"
pred_status = "state_and_forcing"  # specifies what fields are being predicted: "state", "forcing", "state_and_forcing"
if pred_status=="forcing":
    n_unroll = 1    # due to memory limitations
    C_out_total = 2
    pred_residual = False
elif pred_status=="state_and_forcing":
    C_out_total = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load wet mask to cpu
wet_mask_loader = data_loaders.WetMaskFromNetCDF(
    wet_path=wet_mask_path,
    var_name='wet_mask',
    engine="netcdf4"
)
wet = wet_mask_loader.get_wet_mask()  # Shape: (H, W)

wet_mask_nan_loader = data_loaders.WetMaskFromNetCDF(
    wet_path=wet_mask_path,
    var_name='wet_mask_nan',
    engine="netcdf4"
)
wet_nan = wet_mask_nan_loader.get_wet_mask()  # Shape: (H, W)

In [ ]:
# load data statistics (here mean and std that are computed from training data)
stats = np.load(f"data_stats_91_day_sequence_of_{n_unroll}_{pred_status}.npz")
data_mean = stats["mean"]
data_std = stats["std"]
data_mean = torch.from_numpy(data_mean).float()
data_std  = torch.from_numpy(data_std).float()

In [ ]:
# Load learned model
model_adj = model.AdjointModel(
    backbone=model.AdjointNet(wet, in_channels=C_in, out_channels=C_out_total)
).to(device)

checkpoint_save_path = f"checkpoints/checkpoint_91_day_{loss_name}_loss_sequence_of_{n_unroll}_{pred_status}_2.pt"
checkpoint = torch.load(checkpoint_save_path, map_location=torch.device(device))
model_to_load = model_adj.module if hasattr(model_adj, "module") else model_adj
model_to_load.load_state_dict(checkpoint["model_state_dict"])

# Long rollout predictions

In [ ]:
# Load data for specific sample points
path_in_2_years = "/nobackupp17/ifenty/AD_ML/2025-08-30_20pts_long/long_adetan_training_points/etan_ad_2025-08-30_20pts_long_000.nc"
N_total_samples = 20
N_eval_samples = 20
random.seed(42)
sample_indices = random.sample(range(0, N_total_samples), N_eval_samples)   # randomly select 20 sample points
data_in_2_years, sam_i_2_year, sam_j_2_year = load_sample_points(path_in_2_years,
                                                                       sample_indices, wet=wet, device=device, remove_pole=remove_pole)

In [ ]:
idx_t0 = 3
x0 = data_in_2_years[:,idx_t0].clone()  # initial condition at lag t0, shape [N_eval_samples, C_in, H, W]
n_lags = 700 # predict for 700 days backward
C_out_total = C_out
save_or_load = "save"  # "save" or "load"

if save_or_load=="save":
    y_pred = post_processing.generate_adjoint_rollout_from_initial_lag(model_adj, x0, data_mean, data_std, C_out_total, n_lags,
                                                                    wet, pred_residual=pred_residual, remove_pole=remove_pole)
    torch.save(y_pred, f"rollout_pred_91_day_{loss_name}_loss_{n_unroll}_{pred_status}.pt")
    # y_pred has shape [N_eval_samples, n_lags, C_out_total, H, W]
    y_pred *= wet_nan.cpu() # set land cells to NaN

else:
    y_pred = torch.load(f"rollout_pred_91_day_{loss_name}_loss_{n_unroll}_{pred_status}.pt", map_location="cpu")